# QUANTIZATION-AWARE TRAINING (QAT)

    This notebook demonstrates QUANTIZATION-AWARE TRAINING (QAT), an advanced
    quantization technique that simulates quantization DURING training.
    
    KEY DIFFERENCE FROM PTQ (Post-Training Quantization):
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    POST-TRAINING QUANTIZATION (PTQ):
        Train (FP32) → Calibrate → Quantize (INT8)
        ✓ Fast and easy
        ✗ May lose 1-3% accuracy
    
    QUANTIZATION-AWARE TRAINING (QAT):
        Train with simulated quantization → Quantize (INT8)
        ✓ Better accuracy (<1% drop)
        ✓ Model learns to be robust to quantization
        ✗ Requires retraining (more expensive)
    
    THE BIG IDEA: During training, simulate the quantization noise so the model
    learns weights that work well even after being quantized!
    
    Visual comparison:
        
    PTQ: [Train in FP32] -> [Surprise Now you're INT8] -> Accuracy drop
                                ↑
                        Model wasn't prepared!
    
    QAT: [Train with fake quantization] -> [Convert to INT8] -> ✓ Minimal drop
                    ↓                              ↑
            Model adapts during training   Already prepared!

In [1]:
import torch
import torchvision.datasets as datasets 
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import os

# Make torch deterministic for reproducible results
_ = torch.manual_seed(0)

# Normalize MNIST images
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load MNIST training set
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

# Load MNIST test set
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(mnist_testset, batch_size=10, shuffle=True)

# Use CPU (quantization works on CPU)
device = "cpu"